In [24]:
import pandas as pd

In [25]:
df = pd.read_csv("./release_assets/external/DeePiCt/3d_cnn/metadata_dxd20240920_hsp60.csv")
df = df.set_index("tomo_name")
df.head()

,tomo,lamella_file,path_to_motl_clean_ribo,hsp60_mask
tomo_name,,,,
24092001,/virtual_data/t...,NaN,NaN,/virtual_data/t...
24092002,/virtual_data/t...,NaN,NaN,/virtual_data/t...
24092018,/virtual_data/t...,NaN,NaN,/virtual_data/t...
24092024,/virtual_data/t...,NaN,NaN,/virtual_data/t...
24092028,/virtual_data/t...,NaN,NaN,/virtual_data/t...


In [26]:
names = ["24092001", "24092002", "24092018", "24092024", "24092028", "24092031"] 
for i in names:
    df.loc[i, "hsp60_mask"] = f"/virtual_data/test_data/dxd20240920_hsp60_0327_groundtruth/after_class/{i}_processed.mrc"

In [17]:
# The original CSV has 12 rows: 6 rows with tomo paths + 6 duplicate rows with
# only hsp60_mask.  Cell 3 already set hsp60_mask on ALL rows sharing the same
# tomo_name index, so filtering to rows where tomo is not null gives exactly
# the 6 rows needed — each with a tomo path and the correct hsp60_mask.
df_clean = df.reset_index()
df_clean = df_clean[df_clean["tomo"].notna()].reset_index(drop=True)

assert df_clean["tomo_name"].is_unique, "tomo_name still has duplicates!"
assert df_clean["tomo"].notna().all(), "Some rows are missing tomo paths!"
assert df_clean["hsp60_mask"].notna().all(), "Some rows are missing hsp60_mask!"
assert len(df_clean) == 6, f"Expected 6 rows, got {len(df_clean)}"

df_clean.to_csv("./release_assets/external/DeePiCt/3d_cnn/metadata_hsp60.csv", index=False)
print(df_clean[["tomo_name", "tomo", "hsp60_mask"]])

  tomo_name                                               tomo  \
0  24092001  /virtual_data/t...   
1  24092002  /virtual_data/t...   
2  24092018  /virtual_data/t...   
3  24092024  /virtual_data/t...   
4  24092028  /virtual_data/t...   
5  24092031  /virtual_data/t...   

                                          hsp60_mask  
0  /virtual_data/t...  
1  /virtual_data/t...  
2  /virtual_data/t...  
3  /virtual_data/t...  
4  /virtual_data/t...  
5  /virtual_data/t...  


In [ ]:
from pathlib import Path
import ast
import json
import os
import shutil
import subprocess
import time

import pandas as pd
import yaml

repo = Path("./release_assets/external/DeePiCt/3d_cnn")
metadata_csv = repo / "metadata_hsp60_611.csv"
base_out = Path("/virtual_data/train_data/training/deepict/leave_one_out_hsp60")
configs_dir = base_out / "configs"
configs_dir.mkdir(parents=True, exist_ok=True)
pythonpath = repo / "src"
deepict_env = "python"

names = ["24092001", "24092002", "24092018", "24092024", "24092028", "24092031"]

# Reuse the existing DeePiCt hsp60 config as the local hypothesis for the correct pipeline shape.
base_config_path = repo / "configs_hsp60" / "config_hsp60_fold1.yaml"
base_config = yaml.safe_load(base_config_path.read_text())

base_config["dataset_table"] = str(metadata_csv)
base_config["training"]["active"] = True
base_config["training"]["force_retrain"] = True
base_config["prediction"]["active"] = True
base_config["postprocessing_clustering"]["active"] = True
base_config["evaluation"]["segmentation_evaluation"]["active"] = True
base_config["evaluation"]["particle_picking"]["active"] = False

# DeePiCt expects postprocessing_clustering.min_cluster_size.
# Some template configs carry the typo min_clusterls_size; normalize it here.
pp_cfg = base_config.setdefault("postprocessing_clustering", {})
if "min_cluster_size" not in pp_cfg:
    pp_cfg["min_cluster_size"] = pp_cfg.pop("min_clusterls_size", 10)


def run_deepict_config(config_path, gpu_id, log_path):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    env["PYTHONPATH"] = str(pythonpath)
    cmd = [str(repo / "deploy_local.sh"), str(config_path)]
    start = time.time()
    with log_path.open("w") as log_file:
        proc = subprocess.run(
            cmd,
            cwd=repo,
            env=env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            check=False,
            text=True,
        )
    return proc.returncode, round(time.time() - start, 2)


results = []
for holdout_idx, holdout_name in enumerate(names):
    train_names = [name for name in names if name != holdout_name]
    fold_name = f"hsp60_loo_{holdout_idx}_{holdout_name}"

    output_dir = base_out / f"{fold_name}_out"
    work_dir = base_out / f"{fold_name}_work"
    model_path = base_out / f"{fold_name}_model.pth"
    config_path = configs_dir / f"{fold_name}.yaml"
    log_path = base_out / f"{fold_name}.log"

    cfg = json.loads(json.dumps(base_config))
    cfg["output_dir"] = str(output_dir)
    cfg["work_dir"] = str(work_dir)
    cfg["model_path"] = str(model_path)
    cfg["tomos_sets"]["training_list"] = train_names
    cfg["tomos_sets"]["prediction_list"] = [holdout_name]

    if output_dir.exists():
        shutil.rmtree(output_dir)
    if work_dir.exists():
        shutil.rmtree(work_dir)
    if model_path.exists():
        model_path.unlink()

    config_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

    gpu_id = # holdout_idx % 4
    exit_code, duration_sec = run_deepict_config(config_path=config_path, gpu_id=gpu_id, log_path=log_path)

    pp_stats_path = output_dir / "pp_statistics.csv"
    dice_value = None
    if pp_stats_path.exists():
        pp_df = pd.read_csv(pp_stats_path)
        fold_rows = pp_df[
            (pp_df["tomo_name"] == holdout_name)
            & (pp_df["statistic_variable"] == "dice")
        ]
        if not fold_rows.empty:
            dice_value = float(fold_rows.iloc[-1]["statistic_value"])

    results.append(
        {
            "fold_name": fold_name,
            "holdout_name": holdout_name,
            "training_list": train_names,
            "prediction_list": [holdout_name],
            "config_path": str(config_path),
            "output_dir": str(output_dir),
            "work_dir": str(work_dir),
            "model_path": str(model_path),
            "log_path": str(log_path),
            "pp_statistics": str(pp_stats_path),
            "gpu_id": gpu_id,
            "exit_code": exit_code,
            "duration_sec": duration_sec,
            "dice": dice_value,
        }
    )

results_df = pd.DataFrame(results)
results_csv = base_out / "leave_one_out_results.csv"
results_json = base_out / "leave_one_out_results.json"
results_df.to_csv(results_csv, index=False)
results_json.write_text(json.dumps(results, indent=2))

print(results_df[["fold_name", "holdout_name", "exit_code", "duration_sec", "dice"]])
print("\nSaved:")
print(results_csv)
print(results_json)

              fold_name holdout_name  exit_code  duration_sec  dice
0  hsp60_loo_0_24092001     24092001          1        849.07  None
1  hsp60_loo_1_24092002     24092002          0      10478.10  None
2  hsp60_loo_2_24092018     24092018          1        819.98  None
3  hsp60_loo_3_24092024     24092024          0      12148.98  None
4  hsp60_loo_4_24092028     24092028          1        524.96  None
5  hsp60_loo_5_24092031     24092031          0       9916.14  None

Saved:
/virtual_data/train_data/training/deepict/leave_one_out_hsp60/leave_one_out_results.csv
/virtual_data/train_data/training/deepict/leave_one_out_hsp60/leave_one_out_results.json


In [19]:
from pathlib import Path
import json
import pandas as pd
import os
import shutil
import subprocess
import time
import yaml

repo = Path("./release_assets/external/DeePiCt/3d_cnn")
metadata_csv = repo / "metadata_hsp60.csv"
base_out = Path("/virtual_data/train_data/training/deepict/leave_one_out_hsp60")
configs_dir = base_out / "configs"
pythonpath = repo / "src"

names = ["24092001", "24092002", "24092018", "24092024", "24092028", "24092031"]

# Load base config
base_config_path = repo / "configs_hsp60" / "config_hsp60_fold1.yaml"
base_config = yaml.safe_load(base_config_path.read_text())

base_config["dataset_table"] = str(metadata_csv)
base_config["training"]["active"] = True
base_config["training"]["force_retrain"] = True
base_config["prediction"]["active"] = True
base_config["postprocessing_clustering"]["active"] = True
base_config["evaluation"]["segmentation_evaluation"]["active"] = True
base_config["evaluation"]["particle_picking"]["active"] = False

pp_cfg = base_config.setdefault("postprocessing_clustering", {})
if "min_cluster_size" not in pp_cfg:
    pp_cfg["min_cluster_size"] = pp_cfg.pop("min_clusterls_size", 10)

def run_deepict_config(config_path, gpu_id, log_path):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    env["PYTHONPATH"] = str(pythonpath)
    cmd = [str(repo / "deploy_local.sh"), str(config_path)]
    start = time.time()
    with log_path.open("w") as log_file:
        proc = subprocess.run(
            cmd,
            cwd=repo,
            env=env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            check=False,
            text=True,
        )
    return proc.returncode, round(time.time() - start, 2)

# Rerun only the failed folds: indices 0, 2, 4
failed_indices = [0, 2, 4]
rerun_results = []

for holdout_idx in failed_indices:
    holdout_name = names[holdout_idx]
    train_names = [name for name in names if name != holdout_name]
    fold_name = f"hsp60_loo_{holdout_idx}_{holdout_name}"

    output_dir = base_out / f"{fold_name}_out"
    work_dir = base_out / f"{fold_name}_work"
    model_path = base_out / f"{fold_name}_model.pth"
    config_path = configs_dir / f"{fold_name}.yaml"
    log_path = base_out / f"{fold_name}.log"

    cfg = json.loads(json.dumps(base_config))
    cfg["output_dir"] = str(output_dir)
    cfg["work_dir"] = str(work_dir)
    cfg["model_path"] = str(model_path)
    cfg["tomos_sets"]["training_list"] = train_names
    cfg["tomos_sets"]["prediction_list"] = [holdout_name]

    if output_dir.exists():
        shutil.rmtree(output_dir)
    if work_dir.exists():
        shutil.rmtree(work_dir)
    if model_path.exists():
        model_path.unlink()

    config_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

    gpu_id = holdout_idx % 4
    exit_code, duration_sec = run_deepict_config(config_path=config_path, gpu_id=gpu_id, log_path=log_path)

    pp_stats_path = output_dir / "pp_statistics.csv"
    dice_value = None
    if pp_stats_path.exists():
        pp_df = pd.read_csv(pp_stats_path)
        fold_rows = pp_df[
            (pp_df["tomo_name"] == holdout_name)
            & (pp_df["statistic_variable"] == "dice")
        ]
        if not fold_rows.empty:
            dice_value = float(fold_rows.iloc[-1]["statistic_value"])

    rerun_results.append(
        {
            "fold_name": fold_name,
            "holdout_name": holdout_name,
            "training_list": train_names,
            "prediction_list": [holdout_name],
            "config_path": str(config_path),
            "output_dir": str(output_dir),
            "work_dir": str(work_dir),
            "model_path": str(model_path),
            "log_path": str(log_path),
            "pp_statistics": str(pp_stats_path),
            "gpu_id": gpu_id,
            "exit_code": exit_code,
            "duration_sec": duration_sec,
            "dice": dice_value,
        }
    )

# Load existing results and update with rerun results
existing_results = json.loads((base_out / "leave_one_out_results.json").read_text())
for existing in existing_results:
    for rerun in rerun_results:
        if existing["fold_name"] == rerun["fold_name"]:
            existing.update(rerun)
            break

# Save updated results
results_csv = base_out / "leave_one_out_results.csv"
results_json = base_out / "leave_one_out_results.json"
results_df = pd.DataFrame(existing_results)
results_df.to_csv(results_csv, index=False)
results_json.write_text(json.dumps(existing_results, indent=2))

print("Rerun results for failed folds:")
rerun_df = pd.DataFrame(rerun_results)
print(rerun_df[["fold_name", "holdout_name", "exit_code", "duration_sec", "dice"]])
print("\nUpdated aggregate results:")
print(results_df[["fold_name", "holdout_name", "exit_code", "duration_sec", "dice"]])
print("\nSaved:")
print(results_csv)
print(results_json)

Rerun results for failed folds:
              fold_name holdout_name  exit_code  duration_sec  dice
0  hsp60_loo_0_24092001     24092001          1        371.76  None
1  hsp60_loo_2_24092018     24092018          0       8799.72  None
2  hsp60_loo_4_24092028     24092028          1        537.80  None

Updated aggregate results:
              fold_name holdout_name  exit_code  duration_sec  dice
0  hsp60_loo_0_24092001     24092001          1        371.76  None
1  hsp60_loo_1_24092002     24092002          0      10478.10  None
2  hsp60_loo_2_24092018     24092018          0       8799.72  None
3  hsp60_loo_3_24092024     24092024          0      12148.98  None
4  hsp60_loo_4_24092028     24092028          1        537.80  None
5  hsp60_loo_5_24092031     24092031          0       9916.14  None

Saved:
/virtual_data/train_data/training/deepict/leave_one_out_hsp60/leave_one_out_results.csv
/virtual_data/train_data/training/deepict/leave_one_out_hsp60/leave_one_out_results.json


In [20]:
from pathlib import Path
import json
import pandas as pd
import os
import shutil
import subprocess
import time
import yaml

repo = Path("./release_assets/external/DeePiCt/3d_cnn")
metadata_csv = repo / "metadata_hsp60.csv"
base_out = Path("/virtual_data/train_data/training/deepict/leave_one_out_hsp60")
configs_dir = base_out / "configs"
pythonpath = repo / "src"

names = ["24092001", "24092002", "24092018", "24092024", "24092028", "24092031"]

# Load base config
base_config_path = repo / "configs_hsp60" / "config_hsp60_fold1.yaml"
base_config = yaml.safe_load(base_config_path.read_text())

base_config["dataset_table"] = str(metadata_csv)
base_config["training"]["active"] = True
base_config["training"]["force_retrain"] = True
base_config["training"]["batch_size"] = 6  # REDUCED from 12 to fit in GPU memory
base_config["prediction"]["active"] = True
base_config["postprocessing_clustering"]["active"] = True
base_config["evaluation"]["segmentation_evaluation"]["active"] = True
base_config["evaluation"]["particle_picking"]["active"] = False

pp_cfg = base_config.setdefault("postprocessing_clustering", {})
if "min_cluster_size" not in pp_cfg:
    pp_cfg["min_cluster_size"] = pp_cfg.pop("min_clusterls_size", 10)

def run_deepict_config(config_path, gpu_id, log_path):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    env["PYTHONPATH"] = str(pythonpath)
    cmd = [str(repo / "deploy_local.sh"), str(config_path)]
    start = time.time()
    with log_path.open("w") as log_file:
        proc = subprocess.run(
            cmd,
            cwd=repo,
            env=env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            check=False,
            text=True,
        )
    return proc.returncode, round(time.time() - start, 2)

# Retry folds that failed due to GPU memory or other issues
# Fold 0, 2, 4 from previous run: 0 and 4 had OOM, 2 appeared to complete
failed_indices = [0, 2, 4]
rerun_results = []

for holdout_idx in failed_indices:
    holdout_name = names[holdout_idx]
    train_names = [name for name in names if name != holdout_name]
    fold_name = f"hsp60_loo_{holdout_idx}_{holdout_name}"

    output_dir = base_out / f"{fold_name}_out"
    work_dir = base_out / f"{fold_name}_work"
    model_path = base_out / f"{fold_name}_model.pth"
    config_path = configs_dir / f"{fold_name}.yaml"
    log_path = base_out / f"{fold_name}.log"

    cfg = json.loads(json.dumps(base_config))
    cfg["output_dir"] = str(output_dir)
    cfg["work_dir"] = str(work_dir)
    cfg["model_path"] = str(model_path)
    cfg["tomos_sets"]["training_list"] = train_names
    cfg["tomos_sets"]["prediction_list"] = [holdout_name]

    if output_dir.exists():
        shutil.rmtree(output_dir)
    if work_dir.exists():
        shutil.rmtree(work_dir)
    if model_path.exists():
        model_path.unlink()

    config_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

    gpu_id = holdout_idx % 4
    exit_code, duration_sec = run_deepict_config(config_path=config_path, gpu_id=gpu_id, log_path=log_path)

    pp_stats_path = output_dir / "pp_statistics.csv"
    dice_value = None
    if pp_stats_path.exists():
        pp_df = pd.read_csv(pp_stats_path)
        fold_rows = pp_df[
            (pp_df["tomo_name"] == holdout_name)
            & (pp_df["statistic_variable"] == "dice")
        ]
        if not fold_rows.empty:
            dice_value = float(fold_rows.iloc[-1]["statistic_value"])

    rerun_results.append(
        {
            "fold_name": fold_name,
            "holdout_name": holdout_name,
            "training_list": train_names,
            "prediction_list": [holdout_name],
            "config_path": str(config_path),
            "output_dir": str(output_dir),
            "work_dir": str(work_dir),
            "model_path": str(model_path),
            "log_path": str(log_path),
            "pp_statistics": str(pp_stats_path),
            "gpu_id": gpu_id,
            "exit_code": exit_code,
            "duration_sec": duration_sec,
            "dice": dice_value,
        }
    )

# Load existing results and update with rerun results
existing_results = json.loads((base_out / "leave_one_out_results.json").read_text())
for existing in existing_results:
    for rerun in rerun_results:
        if existing["fold_name"] == rerun["fold_name"]:
            existing.update(rerun)
            break

# Save updated results
results_csv = base_out / "leave_one_out_results.csv"
results_json = base_out / "leave_one_out_results.json"
results_df = pd.DataFrame(existing_results)
results_df.to_csv(results_csv, index=False)
results_json.write_text(json.dumps(existing_results, indent=2))

print("Rerun results (batch_size=6):")
rerun_df = pd.DataFrame(rerun_results)
print(rerun_df[["fold_name", "holdout_name", "exit_code", "duration_sec", "dice"]])
print("\nFull aggregate results:")
print(results_df[["fold_name", "holdout_name", "exit_code", "duration_sec", "dice"]])

failed_count = (results_df["exit_code"] != 0).sum()
print(f"\nTotal failed: {failed_count}/6")
if failed_count == 0:
    print("✓ ALL 6 TESTS PASSED!")

Rerun results (batch_size=6):
              fold_name holdout_name  exit_code  duration_sec  dice
0  hsp60_loo_0_24092001     24092001          0       9014.82  None
1  hsp60_loo_2_24092018     24092018          1          4.84  None
2  hsp60_loo_4_24092028     24092028          0       9107.74  None

Full aggregate results:
              fold_name holdout_name  exit_code  duration_sec  dice
0  hsp60_loo_0_24092001     24092001          0       9014.82  None
1  hsp60_loo_1_24092002     24092002          0      10478.10  None
2  hsp60_loo_2_24092018     24092018          1          4.84  None
3  hsp60_loo_3_24092024     24092024          0      12148.98  None
4  hsp60_loo_4_24092028     24092028          0       9107.74  None
5  hsp60_loo_5_24092031     24092031          0       9916.14  None

Total failed: 1/6


In [23]:
import shutil
import subprocess
import os
import time
import json
import yaml
import pandas as pd
from pathlib import Path

repo = Path("./release_assets/external/DeePiCt/3d_cnn")
metadata_csv = repo / "metadata_hsp60.csv"
base_out = Path("/virtual_data/train_data/training/deepict/leave_one_out_hsp60")
configs_dir = base_out / "configs"
pythonpath = repo / "src"
names = ["24092001", "24092002", "24092018", "24092024", "24092028", "24092031"]

holdout_idx  = 2
holdout_name = names[holdout_idx]
train_names  = [n for n in names if n != holdout_name]
fold_name    = f"hsp60_loo_{holdout_idx}_{holdout_name}"

output_dir   = base_out / f"{fold_name}_out"
work_dir     = base_out / f"{fold_name}_work"
model_path   = base_out / f"{fold_name}_model.pth"
model_best   = base_out / f"{fold_name}_model_best.pth"
config_path  = configs_dir / f"{fold_name}.yaml"
log_path     = base_out / f"{fold_name}.log"

# Load config; training is already done for fold 2 — set force_retrain=False
# so Snakemake skips training and goes straight to predict/postprocess/eval.
base_config = yaml.safe_load((repo / "configs_hsp60" / "config_hsp60_fold1.yaml").read_text())
base_config["dataset_table"] = str(metadata_csv)
base_config["training"]["active"] = True
base_config["training"]["force_retrain"] = False  # model already trained
base_config["training"]["batch_size"] = 6
base_config["prediction"]["active"] = True
base_config["postprocessing_clustering"]["active"] = True
base_config["evaluation"]["segmentation_evaluation"]["active"] = True
base_config["evaluation"]["particle_picking"]["active"] = False
pp_cfg = base_config.setdefault("postprocessing_clustering", {})
if "min_cluster_size" not in pp_cfg:
    pp_cfg["min_cluster_size"] = pp_cfg.pop("min_clusterls_size", 10)

cfg = json.loads(json.dumps(base_config))
cfg["output_dir"] = str(output_dir)
cfg["work_dir"]   = str(work_dir)
cfg["model_path"] = str(model_path)
cfg["tomos_sets"]["training_list"]   = train_names
cfg["tomos_sets"]["prediction_list"] = [holdout_name]

# Clean only the prediction output; keep model files and work/testing_data
if output_dir.exists():
    shutil.rmtree(output_dir)
# Ensure model.pth is present (segment.py needs it; training saves _best.pth)
if not model_path.exists() and model_best.exists():
    import shutil as _sh
    _sh.copy2(str(model_best), str(model_path))
    print(f"Restored {model_path.name} from _best.pth")

config_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = str(holdout_idx % 4)
env["PYTHONPATH"] = str(pythonpath)
start = time.time()
with log_path.open("w") as f:
    proc = subprocess.run(
        [str(repo / "deploy_local.sh"), str(config_path)],
        cwd=repo, env=env, stdout=f, stderr=subprocess.STDOUT, check=False, text=True
    )
duration_sec = round(time.time() - start, 2)

pp_stats_path = output_dir / "pp_statistics.csv"
dice_value = None
if pp_stats_path.exists():
    pp_df = pd.read_csv(pp_stats_path)
    rows = pp_df[(pp_df["tomo_name"] == holdout_name) & (pp_df["statistic_variable"] == "dice")]
    if not rows.empty:
        dice_value = float(rows.iloc[-1]["statistic_value"])

print(f"Fold 2 ({holdout_name}): exit={proc.returncode}, duration={duration_sec}s, dice={dice_value}")

# Update aggregate results
existing_results = json.loads((base_out / "leave_one_out_results.json").read_text())
for r in existing_results:
    if r["fold_name"] == fold_name:
        r.update({"exit_code": proc.returncode, "duration_sec": duration_sec,
                  "pp_statistics": str(pp_stats_path), "dice": dice_value})
        break

results_json = base_out / "leave_one_out_results.json"
results_csv  = base_out / "leave_one_out_results.csv"
results_json.write_text(json.dumps(existing_results, indent=2))
pd.DataFrame(existing_results).to_csv(results_csv, index=False)

results_df = pd.DataFrame(existing_results)
print("\nFull aggregate results:")
print(results_df[["fold_name", "holdout_name", "exit_code", "duration_sec", "dice"]])
failed = (results_df["exit_code"] != 0).sum()
print(f"\nTotal failed: {failed}/6")
if failed == 0:
    print("ALL 6 TESTS PASSED!")

Fold 2 (24092018): exit=0, duration=437.25s, dice=None

Full aggregate results:
              fold_name holdout_name  exit_code  duration_sec  dice
0  hsp60_loo_0_24092001     24092001          0       9014.82  None
1  hsp60_loo_1_24092002     24092002          0      10478.10  None
2  hsp60_loo_2_24092018     24092018          0        437.25  None
3  hsp60_loo_3_24092024     24092024          0      12148.98  None
4  hsp60_loo_4_24092028     24092028          0       9107.74  None
5  hsp60_loo_5_24092031     24092031          0       9916.14  None

Total failed: 0/6
ALL 6 TESTS PASSED!
